In [ ]:
import re

In [ ]:
EPS = 'ε'

grammar_text = """
E -> E + T | T
T -> T * F | F
F -> ( E ) | id
"""

literals = {'+': '+', '*': '*', '(': '(', ')': ')'}
classes = [(re.compile(r'[A-Za-z_][A-Za-z0-9_]*|\d+'), 'id')]


def read_grammar(text):
    prods = []

    for line in text.strip().splitlines():
        line = line.strip()

        if not line:
            continue

        lhs, rhs = re.split(r'->|→', line, maxsplit=1)
        lhs = lhs.strip()

        for alt in rhs.split('|'):
            symbols = alt.split()

            if symbols in ([EPS], ['eps']):
                symbols = []

            prods.append((lhs, tuple(symbols)))

    start = prods[0][0]
    aug = start + "'"
    prods.insert(0, (aug, (start,)))

    nonterms = []

    for lhs, _ in prods:
        if lhs not in nonterms:
            nonterms.append(lhs)

    terms = []

    for _, rhs in prods:
        for X in rhs:
            if X not in nonterms and X not in terms:
                terms.append(X)

    return prods, nonterms, terms, aug


prods, nonterms, terms, aug = read_grammar(grammar_text)

print("AUGMENTED GRAMMAR")
print("-" * 40)

for i, (lhs, rhs) in enumerate(prods):
    print(f"({i}) {lhs} → {' '.join(rhs) if rhs else EPS}")

print("\nTerminals    :", ' '.join(terms))
print("Non-terminals:", ' '.join(n for n in nonterms if n != aug))

AUGMENTED GRAMMAR
----------------------------------------
(0) E' → E
(1) E → E + T
(2) E → T
(3) T → T * F
(4) T → F
(5) F → ( E )
(6) F → id

Terminals    : + * ( ) id
Non-terminals: E T F


In [ ]:
def first_of_sequence(sequence, first):
    result = set()

    for X in sequence:

        if X in terms:
            result.add(X)
            return result

        result |= first[X] - {EPS}

        if EPS not in first[X]:
            return result

    result.add(EPS)

    return result


def compute_first():
    first = {A: set() for A in nonterms}

    changed = True

    while changed:

        changed = False

        for A, rhs in prods:

            new = first_of_sequence(rhs, first)

            if not new <= first[A]:
                first[A] |= new
                changed = True

    return first


def compute_follow(first):
    follow = {A: set() for A in nonterms}

    follow[aug].add('$')

    changed = True

    while changed:

        changed = False

        for A, rhs in prods:

            for i, X in enumerate(rhs):

                if X in nonterms:

                    trailer = first_of_sequence(rhs[i + 1:], first)

                    new = trailer - {EPS}

                    if EPS in trailer:
                        new |= follow[A]

                    if not new <= follow[X]:
                        follow[X] |= new
                        changed = True

    return follow

In [ ]:
first = compute_first()
follow = compute_follow(first)

print("FIRST AND FOLLOW SETS")
print("-" * 40)

for A in nonterms:
    if A != aug:
        print(f"FIRST({A})  = {{ {', '.join(sorted(first[A]))} }}")

print()

for A in nonterms:
    if A != aug:
        print(f"FOLLOW({A}) = {{ {', '.join(sorted(follow[A]))} }}")

FIRST AND FOLLOW SETS
----------------------------------------
FIRST(E)  = { (, id }
FIRST(T)  = { (, id }
FIRST(F)  = { (, id }

FOLLOW(E) = { $, ), + }
FOLLOW(T) = { $, ), *, + }
FOLLOW(F) = { $, ), *, + }


In [ ]:
def closure(items):
    result = list(items)
    seen = set(items)

    i = 0

    while i < len(result):

        p, dot = result[i]
        i += 1

        rhs = prods[p][1]

        if dot < len(rhs) and rhs[dot] in nonterms:

            for q, (lhs, _) in enumerate(prods):

                if lhs == rhs[dot] and (q, 0) not in seen:
                    seen.add((q, 0))
                    result.append((q, 0))

    return result


def goto(I, X):
    moved = [
        (p, dot + 1)
        for p, dot in I
        if dot < len(prods[p][1]) and prods[p][1][dot] == X
    ]

    return closure(moved)


def build_item_sets():
    states = [closure([(0, 0)])]

    index = {frozenset(states[0]): 0}

    edges = {}

    i = 0

    while i < len(states):

        I = states[i]

        symbols = []

        for p, dot in I:

            rhs = prods[p][1]

            if dot < len(rhs) and rhs[dot] not in symbols:
                symbols.append(rhs[dot])

        for X in symbols:

            J = goto(I, X)
            key = frozenset(J)

            if key not in index:
                index[key] = len(states)
                states.append(J)

            edges[(i, X)] = index[key]

        i += 1

    return states, edges


states, edges = build_item_sets()

In [ ]:
def item_string(item):
    p, dot = item
    lhs, rhs = prods[p]

    symbols = list(rhs)
    symbols.insert(dot, '•')

    return f"{lhs} → {' '.join(symbols)}"


def print_item_sets():
    print("CANONICAL COLLECTION OF LR(0) ITEMS")
    print("=" * 60)

    for i, I in enumerate(states):

        print(f"\nI{i}:")

        for item in I:
            print("    " + item_string(item))

        moves = [
            f"{X} → I{j}"
            for (s, X), j in edges.items()
            if s == i
        ]

        if moves:
            print("    goto: " + ",  ".join(moves))

    print("\nTotal number of states:", len(states))


print_item_sets()

CANONICAL COLLECTION OF LR(0) ITEMS

I0:
    E' → • E
    E → • E + T
    E → • T
    T → • T * F
    T → • F
    F → • ( E )
    F → • id
    goto: E → I1,  T → I2,  F → I3,  ( → I4,  id → I5

I1:
    E' → E •
    E → E • + T
    goto: + → I6

I2:
    E → T •
    T → T • * F
    goto: * → I7

I3:
    T → F •

I4:
    F → ( • E )
    E → • E + T
    E → • T
    T → • T * F
    T → • F
    F → • ( E )
    F → • id
    goto: E → I8,  T → I2,  F → I3,  ( → I4,  id → I5

I5:
    F → id •

I6:
    E → E + • T
    T → • T * F
    T → • F
    F → • ( E )
    F → • id
    goto: T → I9,  F → I3,  ( → I4,  id → I5

I7:
    T → T * • F
    F → • ( E )
    F → • id
    goto: F → I10,  ( → I4,  id → I5

I8:
    F → ( E • )
    E → E • + T
    goto: ) → I11,  + → I6

I9:
    E → E + T •
    T → T • * F
    goto: * → I7

I10:
    T → T * F •

I11:
    F → ( E ) •

Total number of states: 12


In [ ]:
def build_table():
    action = {}
    goto_table = {}
    conflicts = []

    def put(state, symbol, value):

        old = action.get((state, symbol))

        if old is not None and old != value:
            conflicts.append((state, symbol, old, value))
        else:
            action[(state, symbol)] = value

    for i, I in enumerate(states):

        for p, dot in I:

            lhs, rhs = prods[p]

            if dot < len(rhs):

                X = rhs[dot]

                if X in terms:
                    put(i, X, f"s{edges[(i, X)]}")

            elif p == 0:
                put(i, '$', 'acc')

            else:
                for a in follow[lhs]:
                    put(i, a, f"r{p}")

        for A in nonterms:
            if A != aug and (i, A) in edges:
                goto_table[(i, A)] = edges[(i, A)]

    return action, goto_table, conflicts


action, goto_table, conflicts = build_table()

In [ ]:
def print_table():
    t_cols = terms + ['$']
    n_cols = [n for n in nonterms if n != aug]

    width = max(6, max(len(x) for x in t_cols + n_cols) + 2)

    print("ACTION AND GOTO TABLE")
    print("=" * 60)

    header = (
        "state".ljust(8)
        + "".join(c.center(width) for c in t_cols)
        + "|"
        + "".join(c.center(width) for c in n_cols)
    )

    print(header)
    print("-" * len(header))

    for i in range(len(states)):

        row = str(i).ljust(8)

        row += "".join(
            action.get((i, c), '').center(width) for c in t_cols
        )

        row += "|"

        row += "".join(
            str(goto_table.get((i, c), '')).center(width) for c in n_cols
        )

        print(row)

    print("\ns<i> = shift to state i, r<j> = reduce by production j, acc = accept, blank = error")

    if conflicts:
        print("\nConflicts found, grammar is not SLR(1):")

        for state, symbol, old, new in conflicts:
            print(f"  state {state}, symbol '{symbol}': {old} / {new}")
    else:
        print("\nNo conflicts, grammar is SLR(1)")


print_table()

ACTION AND GOTO TABLE
state     +     *     (     )     id    $   |  E     T     F   
---------------------------------------------------------------
0                     s4          s5        |  1     2     3   
1         s6                           acc  |                  
2         r2    s7          r2          r2  |                  
3         r4    r4          r4          r4  |                  
4                     s4          s5        |  8     2     3   
5         r6    r6          r6          r6  |                  
6                     s4          s5        |        9     3   
7                     s4          s5        |              10  
8         s6               s11              |                  
9         r1    s7          r1          r1  |                  
10        r3    r3          r3          r3  |                  
11        r5    r5          r5          r5  |                  

s<i> = shift to state i, r<j> = reduce by production j, acc = accept, blank = err

In [ ]:
def tokenize(string):
    tokens = []
    i = 0

    sorted_literals = sorted(literals, key=len, reverse=True)

    while i < len(string):

        if string[i].isspace():
            i += 1
            continue

        matched = False

        for pattern, name in classes:

            m = pattern.match(string, i)

            if m:
                tokens.append(name)
                i = m.end()
                matched = True
                break

        if matched:
            continue

        for literal in sorted_literals:

            if string.startswith(literal, i):
                tokens.append(literals[literal])
                i += len(literal)
                matched = True
                break

        if not matched:
            return None

    return tokens


def production_string(p):
    lhs, rhs = prods[p]
    return f"{lhs} → {' '.join(rhs) if rhs else EPS}"


def slr_parse(string, show_trace=True):
    tokens = tokenize(string)

    if tokens is None:
        if show_trace:
            print("Invalid symbol in input")
        return False

    tokens.append('$')

    stack = [0]
    position = 0
    rows = []
    accepted = False

    while True:

        state = stack[-1]
        symbol = tokens[position]

        act = action.get((state, symbol))

        stack_text = ' '.join(map(str, stack))
        input_text = ' '.join(tokens[position:])

        if act is None:
            rows.append((stack_text, input_text, "Error"))
            break

        if act == 'acc':
            rows.append((stack_text, input_text, "Accept"))
            accepted = True
            break

        if act[0] == 's':

            rows.append((stack_text, input_text, f"Shift {act[1:]}"))

            stack.append(symbol)
            stack.append(int(act[1:]))
            position += 1

        else:

            p = int(act[1:])
            lhs, rhs = prods[p]

            rows.append((stack_text, input_text, f"Reduce {production_string(p)}"))

            if rhs:
                del stack[-2 * len(rhs):]

            next_state = goto_table.get((stack[-1], lhs))

            if next_state is None:
                rows.append((' '.join(map(str, stack)), input_text, "Error"))
                break

            stack.append(lhs)
            stack.append(next_state)

    if show_trace:

        w1 = max(len(r[0]) for r in rows) + 2
        w2 = max(len(r[1]) for r in rows) + 2

        print(f"{'Step':<6}{'Stack'.ljust(w1)}{'Input'.ljust(w2)}Action")
        print("-" * (6 + w1 + w2 + 30))

        for n, (s, i, a) in enumerate(rows, 1):
            print(f"{n:<6}{s.ljust(w1)}{i.ljust(w2)}{a}")

    return accepted

In [ ]:
test_cases = [
    "id+id*id",
    "(id+id)*id",
    "id*id+id",
    "((id))",
    "id+",
    "id id",
    "(id",
    "id+*id",
    "()"
]

print("STRING TESTING")
print("-" * 40)

for string in test_cases:

    result = slr_parse(string, show_trace=False)

    print(f"{string:<15} -> {'Accept' if result else 'Reject'}")

STRING TESTING
----------------------------------------
id+id*id        -> Accept
(id+id)*id      -> Accept
id*id+id        -> Accept
((id))          -> Accept
id+             -> Reject
id id           -> Reject
(id             -> Reject
id+*id          -> Reject
()              -> Reject


In [ ]:
print("=" * 60)
print("SLR PARSER")
print("=" * 60)

string = input("Enter input string (example: id+id*id): ").strip()

print()

result = slr_parse(string)

print("\nResult:", "Accept" if result else "Reject")

SLR PARSER
Enter input string (example: id+id*id): id+id*id

Step  Stack                   Input           Action
----------------------------------------------------------------------------
1     0                       id + id * id $  Shift 5
2     0 id 5                  + id * id $     Reduce F → id
3     0 F 3                   + id * id $     Reduce T → F
4     0 T 2                   + id * id $     Reduce E → T
5     0 E 1                   + id * id $     Shift 6
6     0 E 1 + 6               id * id $       Shift 5
7     0 E 1 + 6 id 5          * id $          Reduce F → id
8     0 E 1 + 6 F 3           * id $          Reduce T → F
9     0 E 1 + 6 T 9           * id $          Shift 7
10    0 E 1 + 6 T 9 * 7       id $            Shift 5
11    0 E 1 + 6 T 9 * 7 id 5  $               Reduce F → id
12    0 E 1 + 6 T 9 * 7 F 10  $               Reduce T → T * F
13    0 E 1 + 6 T 9           $               Reduce E → E + T
14    0 E 1                   $               Accept

Re